<a href="https://colab.research.google.com/github/SAINIDHI2005/IDS_GNN_Repo/blob/main/GAT/KNN_GAT_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
DATASET_PATH = "/Users/balji/OneDrive/GNN_IDS/Balanced_6Class_400K"

In [2]:
import os

print(os.listdir(DATASET_PATH))

['BENIGN.csv', 'DDoS.csv', 'DoS.csv', 'Mirai.csv', 'Recon.csv', 'Spoofing.csv']


In [3]:
!pip install -q torch-geometric

In [4]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

import torch
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv

from sklearn.neighbors import kneighbors_graph

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
ATTACK_FOLDERS = [

    "DDOS",
    "DOS",
    "MIRAI",
    "RECON",
    "SPOOFING"
]

In [6]:
benign_df = pd.read_csv(
    os.path.join(
        DATASET_PATH,
        "BENIGN.csv"
    )
)

benign_df = benign_df.sample(
    n=200000,
    random_state=42
)

benign_df["label"] = 0

print(len(benign_df))

200000


In [7]:
ATTACK_FILES = [
    "DDoS",
    "DoS",
    "Mirai",
    "Recon",
    "Spoofing"
]

attack_dfs = []

for attack in ATTACK_FILES:

    attack_df = pd.read_csv(
        os.path.join(
            DATASET_PATH,
            f"{attack}.csv"
        )
    )

    attack_df = attack_df.sample(
        n=40000,
        random_state=42
    )

    attack_df["label"] = 1

    attack_dfs.append(attack_df)

    print(attack, len(attack_df))

DDoS 40000
DoS 40000
Mirai 40000
Recon 40000
Spoofing 40000


In [8]:
malicious_df = pd.concat(
    attack_dfs,
    ignore_index=True
)

print(len(malicious_df))

200000


In [9]:
df = pd.concat(

    [benign_df, malicious_df],

    ignore_index=True
)

df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(df.shape)

(400000, 87)


In [10]:
print(df["label"].value_counts())

label
0    200000
1    200000
Name: count, dtype: int64


In [11]:
FEATURES_30 = [
    # paste your exact 30 features here
     "bidirectional_mean_ps",
    "dst2src_syn_packets",
    "src2dst_min_ps",
    "ip_version",
    "src2dst_max_piat_ms",
    "dst2src_stddev_piat_ms",
    "bidirectional_max_piat_ms",
    "bidirectional_rst_packets",
    "src2dst_max_ps",
    "src2dst_duration_ms",
    "src2dst_bytes",
    "src2dst_ece_packets",
    "bidirectional_max_ps",
    "dst2src_bytes",
    "bidirectional_bytes",
    "dst2src_max_ps",
    "bidirectional_min_ps",
    "bidirectional_syn_packets",
    "dst2src_stddev_ps",
    "bidirectional_ack_packets",
    "protocol",
    "dst_port",
    "bidirectional_mean_piat_ms",
    "dst2src_duration_ms",
    "src2dst_mean_ps",
    "src2dst_min_piat_ms",
    "bidirectional_stddev_ps",
    "src2dst_stddev_piat_ms",
    "bidirectional_cwr_packets",
    "src2dst_psh_packets"
]

X = df[FEATURES_30]

y = df["label"]

In [12]:
print("Number of features:", len(FEATURES_30))

Number of features: 30


In [13]:
missing = [f for f in FEATURES_30 if f not in df.columns]

print("Missing features:", missing)

Missing features: []


In [14]:
print(X.shape)
print(y.shape)

(400000, 30)
(400000,)


In [15]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_val = scaler.transform(
    X_val
)


In [17]:
GRAPH_SIZE = 1000

K = 15

In [18]:
def create_graph_dataset(X, y):

    graphs = []

    num_graphs = len(X) // GRAPH_SIZE

    for i in range(num_graphs):

        start = i * GRAPH_SIZE

        end = start + GRAPH_SIZE

        X_chunk = X[start:end]

        y_chunk = y.iloc[start:end]

        adjacency = kneighbors_graph(

            X_chunk,

            n_neighbors=K,

            mode='connectivity',

            include_self=False
        )

        edge_index = np.array(
            adjacency.nonzero()
        )

        edge_index = torch.tensor(
            edge_index,
            dtype=torch.long
        )

        x_tensor = torch.tensor(
            X_chunk,
            dtype=torch.float
        )

        y_tensor = torch.tensor(
            y_chunk.values,
            dtype=torch.long
        )

        graph = Data(

            x=x_tensor,

            edge_index=edge_index,

            y=y_tensor
        )

        graphs.append(graph)

    return graphs

In [19]:
train_graphs = create_graph_dataset(
    X_train,
    y_train
)

val_graphs = create_graph_dataset(
    X_val,
    y_val
)


In [20]:
train_loader = DataLoader(
    train_graphs,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    shuffle=False
)

In [21]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv


class GATModel(torch.nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_classes,
        heads=4
    ):

        super().__init__()

        self.conv1 = GATv2Conv(
            in_channels,
            hidden_channels // heads,
            heads=heads
        )

        self.conv2 = GATv2Conv(
            hidden_channels,
            hidden_channels // heads,
            heads=heads
        )

        self.conv3 = GATv2Conv(
            hidden_channels,
            (hidden_channels // 2) // heads,
            heads=heads
        )

        self.bn1 = torch.nn.BatchNorm1d(hidden_channels)
        self.bn2 = torch.nn.BatchNorm1d(hidden_channels)
        self.bn3 = torch.nn.BatchNorm1d(hidden_channels // 2)

        self.classifier = torch.nn.Linear(
            hidden_channels // 2,
            num_classes
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.4, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.4, training=self.training)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        x = self.classifier(x)

        return x

In [22]:

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

model = GATModel(
    in_channels=X_train.shape[1],

    hidden_channels=256,
    num_classes=2,
    heads=4
)

model = model.to(device)

print(model)

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.001
)

criterion = torch.nn.CrossEntropyLoss()

cuda
GATModel(
  (conv1): GATv2Conv(30, 64, heads=4)
  (conv2): GATv2Conv(256, 64, heads=4)
  (conv3): GATv2Conv(256, 32, heads=4)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


In [23]:
def validate():

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for graph in val_loader:

            graph = graph.to(device)

            out = model(
                graph.x,
                graph.edge_index
            )

            pred = out.argmax(dim=1)

            correct += (pred == graph.y).sum().item()
            total += graph.y.size(0)

    return correct / total

In [25]:
import os
import time
import torch
import joblib

SAVE_DIR = "/Users/balji/OneDrive/GNN_IDS/Saved_Model"
os.makedirs(SAVE_DIR, exist_ok=True)

best_val_acc = 0.0
best_epoch = 0

start_time = time.time()

EPOCHS = 150

for epoch in range(EPOCHS):


    model.train()

    total_loss = 0

    for graph in train_loader:

        graph = graph.to(device)

        optimizer.zero_grad()

        out = model(
            graph.x,
            graph.edge_index
        )

        loss = criterion(
            out,
            graph.y
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    model.eval()

    train_correct = 0
    train_total = 0

    with torch.no_grad():

        for graph in train_loader:

            graph = graph.to(device)

            out = model(
                graph.x,
                graph.edge_index
            )

            pred = out.argmax(dim=1)

            train_correct += (pred == graph.y).sum().item()
            train_total += graph.y.size(0)

    train_acc = train_correct / train_total

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for graph in val_loader:

            graph = graph.to(device)

            out = model(
                graph.x,
                graph.edge_index
            )

            pred = out.argmax(dim=1)

            val_correct += (pred == graph.y).sum().item()
            val_total += graph.y.size(0)

    val_acc = val_correct / val_total

    if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            os.path.join(
                SAVE_DIR,
                "KNN_GAT.pth"
            )
        )

        joblib.dump(
            scaler,
            os.path.join(
                SAVE_DIR,
                "KNN_scaler.pkl"
            )
        )

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1:03d}"
        f" | Loss {avg_loss:.4f}"
        f" | Train {train_acc:.4f}"
        f" | Validation {val_acc:.4f}"
    )

end_time = time.time()

training_time = end_time - start_time

print("\n" + "="*60)

print("Training Finished Successfully!")

print(f"Best Validation Accuracy : {best_val_acc:.4f}")
print(f"Best Epoch               : {best_epoch}")
print(f"Training Time            : {training_time:.2f} seconds")
print(f"Training Time            : {training_time/60:.2f} minutes")

print("\nBest model saved to:")
print(os.path.join(SAVE_DIR, "KNN_GAT.pth"))

print("\nScaler saved to:")
print(os.path.join(SAVE_DIR, "KNN_scaler.pkl"))

print("="*60)

Epoch 001 | Loss 0.4603 | Train 0.8070 | Validation 0.8065
Epoch 002 | Loss 0.4121 | Train 0.8157 | Validation 0.8142
Epoch 003 | Loss 0.3928 | Train 0.8232 | Validation 0.8230
Epoch 004 | Loss 0.3800 | Train 0.8253 | Validation 0.8255
Epoch 005 | Loss 0.3712 | Train 0.8360 | Validation 0.8360
Epoch 006 | Loss 0.3629 | Train 0.8328 | Validation 0.8312
Epoch 007 | Loss 0.3579 | Train 0.8444 | Validation 0.8434
Epoch 008 | Loss 0.3518 | Train 0.8466 | Validation 0.8461
Epoch 009 | Loss 0.3472 | Train 0.8506 | Validation 0.8499
Epoch 010 | Loss 0.3422 | Train 0.8480 | Validation 0.8467
Epoch 011 | Loss 0.3382 | Train 0.8538 | Validation 0.8527
Epoch 012 | Loss 0.3340 | Train 0.8560 | Validation 0.8558
Epoch 013 | Loss 0.3313 | Train 0.8572 | Validation 0.8566
Epoch 014 | Loss 0.3288 | Train 0.8589 | Validation 0.8589
Epoch 015 | Loss 0.3262 | Train 0.8598 | Validation 0.8599
Epoch 016 | Loss 0.3237 | Train 0.8608 | Validation 0.8603
Epoch 017 | Loss 0.3224 | Train 0.8613 | Validation 0.86

In [26]:
import time
import torch

print("=" * 60)

# Training Time
print(f"Training Time: {training_time:.2f} seconds ({150} epochs)\n")

# GPU Memory
if torch.cuda.is_available():

    allocated = torch.cuda.memory_allocated() / (1024 ** 2)
    reserved = torch.cuda.memory_reserved() / (1024 ** 2)
    max_reserved = torch.cuda.max_memory_reserved() / (1024 ** 2)

    print(f"Allocated GPU Memory: {allocated:.2f} MB")
    print(f"Reserved GPU Memory: {reserved:.2f} MB")
    print(f"Max GPU Memory: {max_reserved:.2f} MB\n")

# Model Parameters
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters: {total_params}")
print(f"Trainable Parameters: {trainable_params}\n")

# Graph Memory
node_memory = (
    graph.x.element_size()
    * graph.x.nelement()
) / (1024 ** 2)

edge_memory = (
    graph.edge_index.element_size()
    * graph.edge_index.nelement()
) / (1024 ** 2)

graph_memory = node_memory + edge_memory

print(f"Node Feature Memory: {node_memory:.2f} MB")
print(f"Edge Memory: {edge_memory:.2f} MB")
print(f"Total Graph Memory: {graph_memory:.2f} MB")

print("=" * 60)

Training Time: 480.73 seconds (150 epochs)

Allocated GPU Memory: 20.99 MB
Reserved GPU Memory: 234.00 MB
Max GPU Memory: 234.00 MB

Total Parameters: 216066
Trainable Parameters: 216066

Node Feature Memory: 0.11 MB
Edge Memory: 0.23 MB
Total Graph Memory: 0.34 MB
